### Fine tuning **YOLO v8**

In your CNN projects, the network output was a single vector of probabilities (one per class).
A detection network like YOLOv8 outputs a list of predictions, where EACH prediction has:
1. a bounding box [x, y, width, height] — where is the object?
2. a confidence score — how sure is the model there's an object here?
3. class probabilities — which class is this object?

The model checks thousands of possible box locations simultaneously and uses NMS (Non-MaximumSuppression) to keep only the best non-overlapping boxes. That's the core difference.


Training from scratch means starting with random weights and learning everything from zero. This requires millions of images and weeks of compute. 

Fine-tuning starts from weights already trained on a large dataset (COCO — 330,000 images, 80 classes), which means the model already knows how to detect edges, textures, shapes, and even people. You then adapt these weights to detect YOUR specific classes (hardhats, vests etc.) using YOUR 2800-image dataset. Fine-tuning reaches good accuracy in ~50 epochs instead of thousands. This is standard practice.

In [1]:
# import and verify GPU
import torch
from ultralytics import YOLO

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\user\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.7.1+cu118
GPU available: True
GPU name: NVIDIA GeForce RTX 4060 Ti
GPU memory: 17.2 GB


In [2]:
# load pretrained YOLO v8 weights
# 'yolov8s.pt' is automatically downloaded from ultralytics on first run (~22MB)

model = YOLO('yolov8s.pt')
print(model.info()) # this shows layers and parameters

YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
(129, 11166560, 0, 28.816844800000002)


earlier we got the error because there was a version mismatch

In [3]:
# training main cell

results = model.train(
data=r'C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\dataset\Construction Site Safety.v27-yolov8.yolov8\data.yaml', # path to your dataset config
epochs=50, # how many full passes through training data
imgsz=640, # resize all images to 640x640 before feeding to model
batch=16, # process 16 images at once (reduce to 8 if GPU OOM error)
project='runs/ppe', # where to save results
name='v1', # subfolder name inside project
patience=10, # stop early if val loss doesn't improve for 10 epochs
optimizer='AdamW', # optimizer — AdamW works better than SGD for fine-tuning
lr0=0.001, # initial learning rate
device=0, # 0 = first GPU; 'cpu' if no GPU
val=True, # run validation after each epoch (essential — shows if overfitting)
save=True, # save weights after training
plots=True, # save training curves as images
)
print('Best weights saved at:', results.save_dir + '/weights/best.pt')

Ultralytics 8.4.56  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\dataset\Construction Site Safety.v27-yolov8.yolov8\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0

TypeError: unsupported operand type(s) for +: 'WindowsPath' and 'str'

training actually completed successfully — the error only happened on the print statement after training finished. The model trained for all 50 epochs and saved the weights.

results.save_dir returns a WindowsPath object (Python's way of representing file paths on Windows), not a plain string. You tried to concatenate it with a string using +, which only works between two strings. A WindowsPath and a string are different types so Python throws a TypeError

In [4]:
# Option 2 — use the / operator which WindowsPath supports natively
print('Best weights saved at:', results.save_dir / 'weights' / 'best.pt')

Best weights saved at: C:\Users\user\Desktop\ML & DL projects\CV Real time Safety system\notebooks\runs\detect\runs\ppe\v1\weights\best.pt


## Note
The weights saved inside the notebooks/ folder because you ran the training notebook from inside that directory. When you get to the deployment phase and need to copy best.pt to the model/ folder, use this exact path:

Code

import shutil
from pathlib import Path

src = Path(results.save_dir) / 'weights' / 'best.pt'
dst = Path('../model/best.pt')
dst.parent.mkdir(exist_ok=True)
shutil.copy(src, dst)
print('Copied to:', dst)